In [1]:
from utils import get_df

df = get_df(additional_columns=["model_history"])
df.head()

,ID,Method,Model,question,Question-only,target_answer,response,is_correct,input_tokens,output_tokens,total_tokens,reasoning,Error Class,Type,model_history
0,chal-736,ReWOO,google/gemma-3-27b-it,Winter is almost here and most animals are mig...,How many more bird families flew away to afric...,27,27.0,True,733.0,0.0,733.0,Plan:\nPlan: Calculate the difference between ...,NaN,Subtraction,[{'plan': {'steps': [['Calculate the differenc...
1,chal-162,ReWOO,google/gemma-3-27b-it,Paige raised 7 goldfish and 12 catfish in the ...,How many fishes disappeared?,4,4.0,True,728.0,0.0,728.0,Plan:\nPlan: Calculate the total number of fis...,NaN,Subtraction,[{'plan': {'steps': [['Calculate the total num...
3,chal-390,ReWOO,google/gemma-3-27b-it,Debby bought 200 water bottles and 256 soda bo...,How many days would the soda bottles last?,64,64.0,True,746.0,0.0,746.0,Plan:\nPlan: Calculate the total number of sod...,NaN,Common-Division,[{'plan': {'steps': [['Calculate the total num...
4,chal-781,ReWOO,google/gemma-3-27b-it,There were 106 dollars in Olivia's wallet. Aft...,How much did she spend at the supermarket?,31,31.0,True,757.0,0.0,757.0,Plan:\nPlan: Calculate the total amount Olivia...,NaN,Subtraction,[{'plan': {'steps': [['Calculate the total amo...
5,chal-575,ReWOO,google/gemma-3-27b-it,Faye was placing her pencils and crayons into ...,How many pencils does she have?,720,720.0,True,667.0,0.0,667.0,Plan:\nPlan: Calculate the total number of pen...,NaN,Multiplication,[{'plan': {'steps': [['Calculate the total num...


In [2]:
df["Error Class"].unique()

array([nan, 'misinterpretation of information',
       'first correct, then wrong reasoning',
       'first correct, then wrong answer',
       'first correct, then endless plans', 'endless plans',
       'overthinking in plan', 'correct plan, incorrect solve',
       'endless loop', 'Format incorrect', 'wrongly misclassified',
       'tool error, Format incorrect', 'repeating equations and plans',
       'repeating loop', 'format error', 'solved despite wrong plans',
       'first right, then incorrect plan',
       'correct, then incorrect answer statement',
       'partially correct plan but correct solve', 'partially correct',
       'endless plan contains correct plan',
       'partially correct plan, incorrect solve',
       'wrongfully misclassified', 'equation number error',
       'equation error', 'schema, solve not based on plan',
       'schema error', 'tool error', 'tool misuse, wrong reasoning',
       'format incorrect, no plan given',
       'format incorrect, no plan g

'Plan: Calculate the number of children remaining on the bus after 68 got off. #E1 = Calculator[36 - 68]\nPlan: Calculate the number of children who got on by subtracting the remaining from 12. #E2 = Calculator[12 - #E1]\nPlan: Calculate the number of children remaining on the bus after 68 got off. #E1 = Calculator[36 - 68]\nPlan: Calculate the number of children who got on by subtracting the remaining from 12. #E2 = Calculator[12 - #E1]\nPlan: Calculate the number of children remaining on the bus after 68 got off. #E1 = Calculator[36 - 68]\nPlan: Calculate the number of children who got on by subtracting the remaining from 12. #E2 = Calculator[12 - #E1]\nPlan: Calculate the number of children remaining on the bus after 68 got off. #E1 = Calculator[36 - 68]\nPlan: Calculate the number of children who got on by subtracting the remaining from 12. #E2 = Calculator[12 - #E1]\nPlan: Calculate the number of children remaining on the bus after 68 got off. #E1 = Calculator[36 - 68]\nPlan: Calc

In [12]:
import re
from typing import List, Tuple


def parse_plan(plan: str) -> List[Tuple[str, str, str, str]]:
    # remove <think> tags
    plan = re.sub(r'<think>.*?</think>', '', plan, flags=re.DOTALL)

    results = []

    # regex for normal Plan lines
    regex_pattern = r"Plan:\s*(.+?)\s*(#E\d+)\s*=\s*(\w+)\[([^\]]+)\]"
    matches = re.findall(regex_pattern, plan)
    results.extend(matches)

    # Find lines like: #E3 = Calculator[#E2 + 46]
    # If not already in results, add them with a default plan
    calc_lines = re.findall(r"(#E\d+)\s*=\s*(\w+)\[([^\]]+)\]", plan)
    known_ids = {eid for _, eid, _, _ in results}
    for eid, func, args in calc_lines:
        if eid not in known_ids:
            # Provide a placeholder or autogenerated plan
            auto_plan = f"(Auto) Compute {eid} using {func}[{args}]"
            results.append((auto_plan, eid, func, args))

    # put to string together
    str_result = ""
    for i in range(len(results)):
        str_result += f"Plan: {results[i][0]} {results[i][1]} = {results[i][2]}[{results[i][3]}]\n"
    if "Plan: Plan: " in str_result:
        str_result = str_result.replace("Plan: Plan: ", "Plan: ")
    return str_result

def parse_plan(plan: str) -> str:
    # remove <think> tags
    plan = re.sub(r'<think>.*?</think>', '', plan, flags=re.DOTALL)

    results: List[Tuple[str, str, str, str]] = []
    defined_ids = set()

    # regex for normal Plan lines
    regex_pattern = r"Plan:\s*(.+?)\s*(#E\d+)\s*=\s*(\w+)\[([^\]]+)\]"
    matches = re.findall(regex_pattern, plan)

    for desc, eid, func, args in matches:
        if eid not in defined_ids:
            results.append((desc.strip(), eid, func, args))
            defined_ids.add(eid)

    # fallback for lines without a "Plan:"
    fallback_pattern = r"(#E\d+)\s*=\s*(\w+)\[([^\]]+)\]"
    fallback_matches = re.findall(fallback_pattern, plan)

    for eid, func, args in fallback_matches:
        if eid not in defined_ids:
            auto_plan = f"(Auto) Compute {eid} using {func}[{args}]"
            results.append((auto_plan, eid, func, args))
            defined_ids.add(eid)

    # Format as a string if needed
    output = ""
    
    for desc, eid, func, args in results:
        output += f"Plan: {desc} {eid} = {func}[{args}]\n"

    if "Plan: Plan: " in output:
        output = output.replace("Plan: Plan: ", "Plan: ")
    if len(output) < 50:
        print(f"Parsed plan: {output.strip()}")
        print(f"Original plan: {plan.strip()}")
    return output.strip()

df["Adjusted Plan"] = df["reasoning"].apply(parse_plan)
df["Adjusted Plan"].iloc[0]

Parsed plan: 
Original plan: <think>
Okay, let's see. Lewis earns $403 each week for 233 weeks. He has to pay $49 every week as rent. The question is how much money he earns during the harvest season.

First, I need to calculate his total earnings without considering the rent. That would be weekly earnings multiplied by the number of weeks. So 403 times 233. But wait, the problem mentions he has to pay rent each week. So maybe I need to subtract the rent from his total earnings? Or is the rent a separate expense? The question is asking how much he earns during the harvest season, so maybe the rent is a deduction. But the problem says "how much money does he earn," which might mean total earnings minus rent. But I need to check the exact wording.

The problem states: "How much money does he earn during harvest season?" So if he earns $403 each week and pays $49 in rent each week, then his net earnings each week would be 403 - 49. But the question is about the total during the harvest se

'Plan: Calculate the difference between the number of bird families that flew to Africa and the number that flew to Asia to determine how many more went to Africa. #E1 = Calculator[62 - 35]'

In [9]:
from math_datasets.generators.rewoo_v1 import PlanExecutor

plan_executor = PlanExecutor()

def execute_plan(adjusted_plan):
    try:
        s = plan_executor.follow_plan(adjusted_plan)
        return s[-1]["solve"]["result"]
    except Exception as e:
        print(f"Error executing plan: {e}")
        print(f"Plan: {adjusted_plan}")
        return None

df["AutoSolve"] = df["Adjusted Plan"].apply(execute_plan)

Error executing plan: list index out of range
Plan: 
Error executing plan: list index out of range
Plan: 
Error executing plan: list index out of range
Plan: 
Error executing plan: list index out of range
Plan: 
Error executing plan: list index out of range
Plan: 
Error executing plan: list index out of range
Plan: 
Error executing plan: list index out of range
Plan: 
Error executing plan: list index out of range
Plan: 
Error executing plan: list index out of range
Plan: 
Error executing plan: list index out of range
Plan: 
Error executing plan: list index out of range
Plan: 
Error executing plan: list index out of range
Plan: 
Error executing plan: list index out of range
Plan: 
Error executing plan: list index out of range
Plan: 
Error executing plan: list index out of range
Plan: 
Error executing plan: list index out of range
Plan: 
Error executing plan: Recursion limit of 25 reached without hitting a stop condition. You can increase the limit by setting the `recursion_limit` config

In [5]:
print(df[df["response"] == "Error occured."].iloc[1]["Adjusted Plan"])

print(df[df["response"] == "Error occured."].iloc[1]["AutoSolve"])

Plan: Calculate the number of children remaining on the bus after 68 got off. #E1 = Calculator[36 - 68]
Plan: Calculate the number of children who got on by subtracting the remaining from 12. #E2 = Calculator[12 - #E1]
44.0


In [6]:
df[df["AutoSolve"].isnull()].iloc[0]["AutoSolve"]

In [7]:
# count how many of the AutoSolve are correct
from math_datasets.datasets import Dataset

def extract_answer(answer):
    if answer is None:
        return None
    return Dataset.extract_answer(answer)

df["target_answer_float"] = df["target_answer"].apply(extract_answer)
df["AutoSolve_float"] = df["AutoSolve"].apply(extract_answer)

df["AutoSolve_correct"] = df["AutoSolve_float"] == df["target_answer_float"]
df["AutoSolve_correct"].value_counts(), df["is_correct"].value_counts()

(AutoSolve_correct
 False    1485
 True     1071
 Name: count, dtype: int64,
 is_correct
 False    1363
 True     1193
 Name: count, dtype: int64)

In [8]:
df.groupby(["Model"]).agg({
    "AutoSolve_correct": "mean",
    "is_correct": "mean",
    "ID": "count"
})

,AutoSolve_correct,is_correct,ID
Model,,,
google/gemma-3-27b-it,0.869718,0.880282,284
ollama/deepseek-r1_1.5b,0.732394,0.866197,284
ollama/gemma3_1b,0.239437,0.246479,284
ollama/llama3.2_1b,0.397887,0.232394,284
ollama/qwen2-math_1.5b,0.014085,0.010563,284
ollama/qwen2.5_0.5b,0.253521,0.285211,284
ollama/qwen3_0.6b,0.500000,0.59507,284
ollama/qwen3_1.7b,0.732394,0.933099,284
ollama/smollm2_360m,0.031690,0.151408,284
